In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from algbench import read_as_pandas
from tspn_bnb2.misc.paper_style import FULLWIDEFIGURE, init_params
from tspn_bnb2.schemas import AnnotatedInstance, AnnotatedSolution

init_params()

In [ ]:
def read_row(row):
    instance = AnnotatedInstance.model_validate_json(
        row["parameters"]["args"]["kwargs"]["instance_json"]
    )
    solution = AnnotatedSolution.model_validate_json(row["result"]["solution"])

    return {
        "solution": solution,
        "upper_bound": solution.upper_bound,
        "lower_bound": solution.lower_bound,
        "gap": solution.alg_relative_gap,
        "is_optimal": solution.is_optimal,
        "instance_name": row["parameters"]["args"]["kwargs"]["instance_name"],
        "instance": instance,
        "solve_time": row["result"]["solve_time"],
        "n": instance.num_polygons(),
        "decomposition": row["parameters"]["args"]["alg_params"]["decomposition"],
        "instance_type": instance.derive_instance_type(),
    }


benchmark = read_as_pandas("results_decomposition_strategy", read_row)
print("Loaded", len(benchmark), "runs")
benchmark = benchmark.sort_values(by=["instance_type", "decomposition"])

In [ ]:
# validate that solutions are correct.
n_checks = 0
for _, row in benchmark.iterrows():
    solution = row["solution"]
    if solution is None:
        continue
    same_instance = benchmark[benchmark["instance_name"] == row["instance_name"]]

    for _, other in same_instance.iterrows():
        if other["solution"] is None:
            continue
        check = solution.plausibility_check(
            other["solution"],
            eps=0.01,
        )
        if not check["valid"]:
            print(
                f"Check failed for {row['instance_name']}"
                f" {check} {row['solution']}\n"
                f"{row['decomposition']}\n"
                f"{other['solution']}\n"
                f"{other['decomposition']}"
            )
        else:
            n_checks += 1

print(n_checks, "checks succeeded")

In [ ]:
benchmark.groupby(["decomposition", "instance_type"])["is_optimal"].value_counts()

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=FULLWIDEFIGURE)

ax = axs[0]
sns.boxplot(
    benchmark,
    x="decomposition",
    y="solve_time",
    hue="instance_type",
    palette=sns.color_palette(),
    ax=ax,
    hue_order=["OSM", "tessellation", "random"],
)

xticks = list(ax.get_xticks())
xticklabels = []

for label in ax.get_xticklabels():
    decomp = label.get_text() != "False"

    solutions_for_label = benchmark[(benchmark["decomposition"] == decomp)]
    optimal_percentage = len(solutions_for_label[solutions_for_label["is_optimal"]]) / len(
        solutions_for_label
    )
    label = "decomposition\nbranching" if decomp else "indicator\nmodeling"
    label += f"\n[{round(optimal_percentage * 100, 2)}\\%]"
    xticklabels.append(label)

ax.set_xticks(xticks)
ax.set_xticklabels(xticklabels)

ax.set_title("lower is better")
ax.set_xlabel("decomposition")
ax.set_ylabel("runtime [s]")


instances_with_both_opt = [
    inst
    for inst in benchmark["instance_name"].unique()
    if len(benchmark[(benchmark["instance_name"] == inst)]) == 2
]
bench = benchmark[benchmark["instance_name"].isin(instances_with_both_opt)]

df = {"instance": [], "instance_type": [], "relative_runtime": []}

for _, row in bench[bench["decomposition"]].iterrows():
    other = bench[(~bench["decomposition"]) & (bench["instance_name"] == row["instance_name"])]
    assert len(other) == 1, len(other)
    other = other.iloc[0]

    df["instance"].append(row["instance_name"])
    df["instance_type"].append(row["instance_type"])
    df["relative_runtime"].append(row["solve_time"] / other["solve_time"])

df = pd.DataFrame(df)
df = df.sort_values(
    "instance_type",
    key=lambda x: pd.Categorical(x, categories=["OSM", "tessellation", "random"], ordered=True),
)

print(len(df))
sns.boxplot(
    df,
    x="instance_type",
    y="relative_runtime",
    hue="instance_type",
    ax=axs[1],
    hue_order=["OSM", "tessellation", "random"],
)

axs[1].set_ylabel("relative runtime [s]")
axs[1].set_title("lower is better for DB")
axs[1].set_xlabel("")
axs[0].set_xlabel("")

fig.legend(loc="outside lower center", ncol=3, bbox_to_anchor=(0.5, -0.32))
axs[0].legend().remove()
fig.subplots_adjust(wspace=0.3)
fig.savefig("../plots/rq3_decomposition_branching/runtimes_all.pdf", bbox_inches="tight")

In [ ]:
def cactus_dataframe(
    data: pd.DataFrame,
    runtime_column: str,
    strategy_column: str,
    instance_column: str,
    flat_line_to: float | None = None,
) -> pd.DataFrame:
    """
    Convert benchmark results into a dataframe suitable for cactus plots.

    Returns a tidy dataframe with columns:
        strategy, runtime, solved
    which can be plotted with seaborn.lineplot(drawstyle="steps-post").
    """

    rows = []

    for strategy, group in data.groupby(strategy_column):
        runtimes = (
            group.groupby(instance_column)[runtime_column].min().dropna().sort_values().to_numpy()
        )

        if len(runtimes) == 0:
            continue

        solved = np.arange(1, len(runtimes) + 1)

        x = runtimes
        y = solved

        if flat_line_to is not None and runtimes[-1] < flat_line_to:
            x = np.append(x, flat_line_to)
            y = np.append(y, solved[-1])

        rows.extend(
            {
                strategy_column: strategy,
                "runtime": xi,
                "solved": yi,
            }
            for xi, yi in zip(x, y)
        )

    return pd.DataFrame(rows)

In [ ]:
instances_with_different_counts = set()
instances_with_no_explored = set()

for _, row in benchmark[benchmark["decomposition"]].iterrows():
    other_entry = benchmark[
        (~benchmark["decomposition"]) & (benchmark["instance_name"] == row["instance_name"])
    ]

    assert len(other_entry) == 1, len(other_entry)
    other_entry = other_entry.iloc[0]

    if (
        row["solution"].statistics["num_explored"]
        != other_entry["solution"].statistics["num_explored"]
    ):
        instances_with_different_counts.add(row["instance_name"])
        assert row["is_optimal"] == other_entry["is_optimal"]
    else:
        instances_with_no_explored.add(row["instance_name"])

print(len(instances_with_different_counts), len(instances_with_no_explored))

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=FULLWIDEFIGURE)


# instance_count = len(set(benchmark["instance_name"].unique()))

bm = benchmark[
    (benchmark["is_optimal"]) & (benchmark["instance_name"].isin(instances_with_different_counts))
]

df = cactus_dataframe(
    bm,
    instance_column="instance_name",
    strategy_column="decomposition",
    runtime_column="solve_time",
    # flat_line_to=benchmark["solve_time"].max(),
)


sns.lineplot(
    ax=axs[0],
    data=df,
    x="runtime",
    y="solved",
    hue="decomposition",
    linestyle="-",
    drawstyle="steps-post",
)

axs[0].set_xlim(0, axs[0].get_xlim()[1])


other_bm = benchmark[
    (~benchmark["is_optimal"]) & (benchmark["instance_name"].isin(instances_with_different_counts))
]

print(len(other_bm))
sns.boxplot(other_bm, x="decomposition", y="gap", ax=axs[1])
sns.stripplot(
    other_bm, x="decomposition", y="gap", ax=axs[1], hue="instance_name", palette="tab20", s=4
)
axs[1].legend().remove()


assert len(bm[bm["decomposition"]]) == len(bm[~bm["decomposition"]])
assert len(other_bm[other_bm["decomposition"]]) == len(other_bm[~other_bm["decomposition"]])
assert len(bm["instance_name"].unique()) == len(bm[bm["decomposition"]])
assert len(other_bm["instance_name"].unique()) == len(other_bm[other_bm["decomposition"]])


axs[0].set_title(f"{len(bm['instance_name'].unique())} instances solved to optimality")
axs[1].set_title(f"{len(other_bm['instance_name'].unique())} instances with gap")

# fig.subplots_adjust(right=0.74)
fig.savefig("../plots/rq3_decomposition_branching/cactus.pdf", bbox_inches="tight")

In [ ]:
benchmark.iloc[2].solution.statistics

In [ ]:
fig, ax = plt.subplots()

benchmark["num_explored"] = benchmark["solution"].apply(lambda r: r.statistics["num_explored"])

sns.boxplot(
    benchmark[benchmark["instance_name"].isin(instances_with_different_counts)],
    x="decomposition",
    y="num_explored",
    ax=ax,
)